# Extracting with Vision Models

**Lesson 8 — Module 1: Extraction**

So far we've extracted text from PDFs using three approaches:
- **PyMuPDF** -- reads the embedded text layer (~300 PDFs/sec, free)
- **Tesseract** -- re-OCRs the page image (~1 PDF/sec, free)
- **Docling** -- combined package with layout analysis and OCR (~1 PDF/sec, free)

All of these work at the character level -- they recognize shapes and map them to text. Vision-capable LLMs take a fundamentally different approach: they **understand** the page image, interpreting layout, context, abbreviations, and even handwriting.

This is the most capable extraction method available -- and also the most expensive and slowest.

## What you'll learn

- How to send a PDF page to a vision model for text extraction
- Where vision models genuinely outperform OCR
- The tradeoffs: cost, speed, hallucination risk, and non-determinism
- When to use vision models in a multi-tier pipeline

## 1. Install Dependencies

In [22]:
%pip install openai pymupdf python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 2. Imports & Configuration

In [23]:
import base64
import os
import subprocess
import tempfile
import time
from pathlib import Path

import pymupdf
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Verify the API key is available before we get deep into the notebook
if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found. Create a .env file with your key:\n"
        "  OPENAI_API_KEY=sk-..."
    )

client = OpenAI()

PDF_DIR = Path("enron_pdfs")
SAMPLE_PDF = PDF_DIR / "E61D04918.pdf"    # moderate OCR errors (Bate:, ENRON CORE.)
EMPTY_PDF = PDF_DIR / "E0033CF3B.pdf"     # no text layer at all
SEVERE_PDF = PDF_DIR / "ECD0D46C3.pdf"    # severe OCR errors (COMP DDENTIAL, EMURCN CURD.)

print(f"PDF directory: {PDF_DIR.resolve()}")

PDF directory: /Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/enron_pdfs


## 3. Sending a Page to a Vision Model

The process is straightforward: render the PDF page to an image, encode it as base64, and send it to the model with a prompt asking for the text content.

In [24]:
EXTRACT_PROMPT = (
    "Extract all text from this document image. "
    "Return only the text content, preserving the original structure."
)

def extract_with_vision(pdf_path, page_num=0, dpi=200, model="gpt-4.1-mini-2025-04-14"):
    """Render a PDF page to an image and extract text with a vision model."""
    doc = pymupdf.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    img_bytes = pix.tobytes("png")
    img_b64 = base64.b64encode(img_bytes).decode()
    doc.close()

    response = client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": EXTRACT_PROMPT},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/png;base64,{img_b64}"
                }},
            ],
        }],
    )

    return response.choices[0].message.content, response.usage

## 4. Vision on a degraded scan

Let's start with `E61D04918.pdf` -- the file with moderate OCR errors (`Bate:` for `Date:`, `ENRON CORE.` for `ENRON CORP.`). The existing OCR text layer has errors throughout. How does a vision model compare?

In [25]:
start = time.perf_counter()
vision_text, usage = extract_with_vision(SAMPLE_PDF)
elapsed = time.perf_counter() - start

print(f"Time: {elapsed:.1f}s | Tokens: {usage.total_tokens}")
print()
print(vision_text[:1200])

Time: 4.9s | Tokens: 2710

CONFIDENTIAL  
Enron Corp.  
Case No. EC-2002-01038  
Doc No. E61D04918  
Date: 01/15/2003  
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.  
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.  
RELEASE IN PART  

From:          Kerri Thompson <kerri.thompson@enron.com>  
Sent:              Wed, 22 Nov 2000 05:41:00 -0800 (PST)  
To:                 Kate Symes <kate.symes@enron.com>  
Subject:         natsource checkout  

465810  
matt  
strike price 70.00  
8.00 premium  

465556  
bob  
broker has mid c  

CONFIDENTIAL  
Enron Corp.  
Case No. EC-2002-01038  
Doc No. E61D04918  
Date: 01/15/2003  
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.  
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.  
ENRON-524674254886


Compare this to the other methods on the same file:
- **PyMuPDF** returned the existing garbled text layer: `Bate:`, `ENRON CORE.`, `EROTECTIVE`
- **Tesseract re-OCR** on the clean underlying image corrected most errors: `Date:`, `ENRON CORP.`, `PROTECTIVE`
- **Docling** with forced OCR also produced clean output from the same image

The vision model reads the page as a human would -- interpreting the image directly rather than recognising individual characters. On this file, re-OCR already corrects the errors. The vision model's advantage is more apparent on truly degraded images where even re-OCR struggles.

## 5. Image-Only PDF

Vision models handle image-only PDFs (no text layer) just as easily — the page image is all they need.

In [26]:
start = time.perf_counter()
empty_text, usage = extract_with_vision(EMPTY_PDF)
elapsed = time.perf_counter() - start

print(f"Time: {elapsed:.1f}s | Tokens: {usage.total_tokens}")
print()
print(empty_text[:1000])

Time: 15.2s | Tokens: 3056

CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0033CF3B
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART

From:              Schedule Crawler <pete.davis@enron.com>
Sent:                Mon, 9 Apr 2001 01:41:00 -0700 (PDT)
To:                   pete.davis@enron.com
Cc:                    bert.meyers@enron.com, bill.williams.III@enron.com,
                       Craig.Dean@enron.com, dporter3@enron.com, Eric.Linder@enron.com,
                       Geir.Solberg@enron.com, jbryson@enron.com, leaf.harasin@enron.com,
                       monika.causholli@enron.com, mark.guzman@enron.com,
                       pete.davis@enron.com, ryan.slinger@enron.com
Subject:          Start Date: 4/9/01; HourAhead hour: 9; <CODESITE>

Start Date: 4/9/01; HourAhead hour: 9;   No ancillary schedules awarded.
Variances detected.
Variances detected in SC Trades schedule.


Clean output — correct doc number, proper email formatting, natural body text. The vision model doesn't care whether a text layer exists; it works directly from the page image.

## 6. Quality comparison

Let's put all four methods side-by-side on the same PDF.

In [27]:
# PyMuPDF get_text()
doc = pymupdf.open(SAMPLE_PDF)
page = doc[0]
pymupdf_text = page.get_text()

# Tesseract CLI at 300 DPI
pix = page.get_pixmap(dpi=300)
with tempfile.NamedTemporaryFile(suffix=".png") as f:
    pix.save(f.name)
    result = subprocess.run(
        ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
        capture_output=True, text=True,
    )
    tesseract_text = result.stdout
doc.close()

# Show first ~400 chars of each
for label, text in [
    ("PyMuPDF get_text() -- existing OCR layer", pymupdf_text),
    ("Tesseract CLI -- re-OCR at 300 DPI", tesseract_text),
    ("Vision LLM -- gpt-4.1-mini", vision_text),
]:
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(text[:400])
    print()

PyMuPDF get_text() -- existing OCR layer
CONFIDENTIAL
Enron Corp.
Case No. EC-26002-016038
Doc No. Es6lbo4918
Bate: O1/15/2003
ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Kerri Thompson <kerri.thompson@enron,.
com>
Sent:
Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To:
Kate Symes <kate.symes@enron.com>
Subject :
natsource checkout
465810
matt
strike price

Tesseract CLI -- re-OCR at 300 DPI
CONFIDENTIAL

Enron Corp.

Case No. EC-2002-01038

Doc No. E61D04918

Date: 01/15/2003

ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART

From: Kerri Thompson <kerri.thompson@enron.com>
Sent: Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To: Kate Symes <kate.symes@enron.com>
Subject: natsource checkout

465810

matt

strike p

Vision LLM -- gpt-4.1-mini
CONFIDENTIAL  
Enron Corp.  
Case No. EC-2002-01038  
Doc No. E61D04918  
Date: 01/15/2003  


### Observations

Compare the header fields across all three outputs. The vision model reads the page as a whole -- understanding context, layout, and likely content -- rather than recognising one character at a time.

On clean B&W scans, all methods perform well. On degraded scans, the vision model's contextual understanding gives it an advantage that character-level OCR can't match. But check for hallucinations too -- the model may "correct" something that was actually right in the original.

## 7. The Tradeoffs

Vision models produce the best quality, but they come with real costs.

### Cost

| Method | Cost per 1,000 pages | Speed |
|---|---|---|
| PyMuPDF `get_text()` | ~$0 | ~300 pages/sec |
| Tesseract OCR | ~$0 | ~1 page/sec |
| Docling | ~$0 | ~1-3 pages/sec |
| Vision LLM (gpt-4.1-mini) | ~$1-5 | <1 page/sec |

At 5,000 pages, that's the difference between free and $5-25. At scale, vision extraction is a targeted tool, not a bulk processing strategy.

### Hallucination risk

The same capability that makes vision models powerful -- understanding context -- also introduces a unique risk: **the model can confidently produce text that isn't on the page**.

- It might "correct" an unusual spelling that was actually accurate
- It might fill in a word it can't quite read, guessing wrong
- It might reformat or reorder content based on what it thinks the structure should be

OCR engines make character-level errors (`Bate:` for `Date:`, `CORE.` for `CORP.`), but they don't invent content. Vision models can -- and the errors look plausible, making them harder to catch.

In our tests, `gpt-4.1-mini` performed well on this dataset. But always verify critical fields (document IDs, dates, names) against the source image when accuracy matters.

### Non-determinism

Run the same page through a vision model twice and you may get slightly different output — different line breaks, different whitespace, occasionally different word choices. Traditional extraction methods (PyMuPDF, Tesseract) produce identical output every time.

For an entity extraction pipeline this usually doesn't matter for natural language fields — downstream parsing handles minor variations. But for identifiers, codes, and exact values, non-determinism can be a problem. If you run this notebook multiple times, you may see different readings for fields like the Doc No.

## 8. Speed Comparison

Let's measure the actual speed difference on a small sample.

In [28]:
# 5 scanned PDFs
SIZE_THRESHOLD = 50_000
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
scanned_pdfs = [p for p in pdf_files if p.stat().st_size >= SIZE_THRESHOLD]
sample = scanned_pdfs[:5]
n = len(sample)

# Tesseract CLI (page 0 only)
start = time.perf_counter()
for p in sample:
    doc = pymupdf.open(p)
    pix = doc[0].get_pixmap(dpi=300)
    with tempfile.NamedTemporaryFile(suffix=".png") as f:
        pix.save(f.name)
        subprocess.run(
            ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
            capture_output=True, text=True,
        )
    doc.close()
elapsed_tesseract = time.perf_counter() - start

# Vision model (page 0 only)
start = time.perf_counter()
for p in sample:
    _, _ = extract_with_vision(p)
elapsed_vision = time.perf_counter() - start

print(f"{'Method':<30} {'Time':>8} {'Pages/sec':>10}")
print("-" * 50)
print(f"{'Tesseract CLI (300 DPI)':<30} {elapsed_tesseract:>7.1f}s {n/elapsed_tesseract:>9.1f}")
print(f"{'Vision model (gpt-4.1-mini)':<30} {elapsed_vision:>7.1f}s {n/elapsed_vision:>9.1f}")
print(f"\nVision is ~{elapsed_vision/elapsed_tesseract:.0f}x slower than Tesseract")

Method                             Time  Pages/sec
--------------------------------------------------
Tesseract CLI (300 DPI)            6.2s       0.8
Vision model (gpt-4.1-mini)       48.6s       0.1

Vision is ~8x slower than Tesseract


## 9. When to use vision models

Vision models are the most capable extraction tool available. They make sense as a **targeted fallback** in a multi-tier strategy:

1. **PyMuPDF** `get_text()` for files with existing text layers (84% of our dataset)
2. **Tesseract** for image-only pages that need OCR (16%)
3. **Vision model** for the truly difficult cases -- degraded scans, handwriting, complex layouts that OCR can't handle

This keeps costs low while ensuring you don't lose data from the hardest documents.

For this workshop, the Enron emails extract well with PyMuPDF + Tesseract. We won't use vision models for the full corpus -- but they're the quality ceiling when other methods fall short.

> **Note:** if you don't have an OpenAI API key, you can skip this notebook. The dataset extracts well with PyMuPDF and OCR alone.

## 10. Corrections instead of regeneration

The process above sends the entire page image to the LLM and asks it to return all the text. That works, but it's expensive -- the model regenerates every word, including the ones that were already correct.

A more efficient approach: send the LLM the existing OCR text alongside the page image, and ask it to return only the corrections. The model reads the full page but writes far less, and you apply the fixes to the original.

The cell below encodes each line of the OCR text with a number, sends the numbered text and the page image to the model, and asks it to return only the line numbers that need correcting -- with the corrected text.

In [29]:
# Encode each line of the OCR text with a number
doc = pymupdf.open(SAMPLE_PDF)
page = doc[0]
bad_text = page.get_text()

lines = bad_text.split("\n")
numbered = {str(i): line for i, line in enumerate(lines) if line.strip()}

# Build the codebook for the prompt
codebook = "\n".join(f"{num} = {line}" for num, line in numbered.items())

print(f"{len(numbered)} lines encoded")
print()
print("Codebook (first 15 lines):")
for num, line in list(numbered.items())[:15]:
    print(f"  {num} = {line}")


34 lines encoded

Codebook (first 15 lines):
  0 = CONFIDENTIAL
  1 = Enron Corp.
  2 = Case No. EC-26002-016038
  3 = Doc No. Es6lbo4918
  4 = Bate: O1/15/2003
  5 = ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
  6 = SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
  7 = RELEASE IN PART
  8 = From:
  9 = Kerri Thompson <kerri.thompson@enron,.
  10 = com>
  11 = Sent:
  12 = Wed, 22 Nov 2000 05:41:00 -0800 (PST)
  13 = To:
  14 = Kate Symes <kate.symes@enron.com>


Each line of the OCR text now has a number. The prompt sends this codebook alongside the page image and asks the model to return only the lines that contain errors — with the corrected text.


In [30]:
# Render the page image
pix = page.get_pixmap(dpi=200)
img_b64 = base64.b64encode(pix.tobytes("png")).decode()

CORRECTIONS_PROMPT = f"""I have OCR text from this page image. Each line is numbered.

Current OCR text:
{codebook}

Compare the OCR text to the page image. Return ONLY the lines that contain errors,
with the corrected text. Use this exact format:

LINE_NUMBER: corrected text

If a line is correct, do not include it. Only return lines that need fixing."""

response = client.chat.completions.create(
    model="gpt-4.1-mini-2025-04-14",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": CORRECTIONS_PROMPT},
            {"type": "image_url", "image_url": {
                "url": f"data:image/png;base64,{img_b64}"
            }},
        ],
    }],
)

corrections_raw = response.choices[0].message.content
print("Corrections returned:")
print(corrections_raw)


Corrections returned:
2: Case No. EC-2002-01038
3: Doc No. E61D04918
4: Date: 01/15/2003
5: ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
6: SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
9: Kerri Thompson <kerri.thompson@enron.com>
15: Subject:
19: strike price 70.00
20: 8.00 premium
23: broker has mid c
25: Enron Corp.
26: Case No. EC-2002-01038
27: Doc No. E61D04918
28: Date:
29: 01/15/2003
30: ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
31: SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
33: ENRON-524674254886


The model returns only the lines that differ from the image — not the entire document. Each correction is a line number and the fixed text.

The next cell applies the corrections back to the original OCR text.


In [31]:
# Parse corrections and apply to original text
corrected = dict(numbered)

for line in corrections_raw.strip().split("\n"):
    # Match 'NUMBER: corrected text' format
    if ":" in line:
        num, _, text = line.partition(":")
        num = num.strip()
        text = text.strip()
        if num in corrected and text:
            corrected[num] = text

# Rebuild the full text
final_text = "\n".join(corrected[num] for num in sorted(corrected, key=int))

# Show the before/after on the header fields
print("=== Key fields: before and after ===")
print()
fields_to_check = ['Case No', 'Doc No', 'Bate:', 'Date:', 'ENRON', 'SUBJECT', 'EROTECTIVE', 'PROTECTIVE', 'thompson']
for num in sorted(corrected, key=int):
    orig = numbered.get(num, '')
    fixed = corrected[num]
    if orig != fixed:
        print(f"  Line {num}:")
        print(f"    Before: {orig}")
        print(f"    After:  {fixed}")


=== Key fields: before and after ===

  Line 2:
    Before: Case No. EC-26002-016038
    After:  Case No. EC-2002-01038
  Line 3:
    Before: Doc No. Es6lbo4918
    After:  Doc No. E61D04918
  Line 4:
    Before: Bate: O1/15/2003
    After:  Date: 01/15/2003
  Line 5:
    Before: ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
    After:  ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
  Line 6:
    Before: SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
    After:  SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
  Line 9:
    Before: Kerri Thompson <kerri.thompson@enron,.
    After:  Kerri Thompson <kerri.thompson@enron.com>
  Line 15:
    Before: Subject :
    After:  Subject:
  Line 23:
    Before: broker has midc
    After:  broker has mid c
  Line 25:
    Before: Enron Gerp.
    After:  Enron Corp.
  Line 26:
    Before: Case Ro. 20-7 002-91078
    After:  Case No. EC-2002-01038
  Line 27:
    Before: wea No. RETDC4I9le2
    After:  Doc No. E61D04

The model corrected `Bate:` to `Date:`, `ENRON CORE.` to `ENRON CORP.`, `EROTECTIVE` to `PROTECTIVE`, and fixed the garbled case and doc numbers -- while leaving every correct line untouched.

But look at line 33: `EKRON-S206 741254888` became `ENRON-524674254886`. The model produced a plausible-looking Enron identifier, but there's no way to know if `524674254886` is the real number -- it likely isn't. This is a hallucinated correction: the model fills in what it can't read with something that looks right.

The corrections approach saves tokens and gets most things right, but it can silently introduce wrong values in fields where the original was too garbled to recover. Use it for body text where small errors are tolerable, not for identifiers or dates where accuracy matters.

In [32]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4.1-mini")

# Full regeneration: the vision model returns the entire document
full_text, full_usage = extract_with_vision(SAMPLE_PDF)
full_tokens = len(enc.encode(full_text))

# Corrections only: the model returns just the changed lines
corrections_tokens = len(enc.encode(corrections_raw))

print("Output token comparison:")
print(f"  Full regeneration: {full_tokens} tokens")
print(f"  Corrections only:  {corrections_tokens} tokens")
print(f"  Saving:            {full_tokens - corrections_tokens} tokens ({(1 - corrections_tokens/full_tokens)*100:.0f}%)")


Output token comparison:
  Full regeneration: 250 tokens
  Corrections only:  209 tokens
  Saving:            41 tokens (16%)


Output tokens are typically more expensive than input tokens. By asking the model to return only the corrections, you reduce output significantly — the model reads the whole page (same input cost) but writes far less.

At scale, across thousands of documents where most lines are correct, this approach can cut vision model costs substantially.

Again, however, if your files are highly corrupted, this will likely lead to more expense, not less.


## Summary

- Vision LLMs **understand** page content, not just character shapes -- the highest quality extraction available
- They handle handwriting, complex layouts, degraded scans, and context recovery that no OCR engine can match
- The tradeoffs are real: **cost** (~$1-5 per 1,000 pages with gpt-4.1-mini), **speed** (<1 page/sec), **hallucination risk** (the model can invent text), and **non-determinism**
- Best used as a **targeted fallback** -- not a primary extraction tool
- A practical tiered strategy: PyMuPDF -> Tesseract -> vision model (only for the hardest cases)

**Next:** You've seen all four extraction approaches. Now you'll run the tiered pipeline on the full dataset and produce the text files that Module 2 will parse.